In [0]:
# Databricks notebook source
from pyspark.sql.functions import lit
# ==========================================
# 1️⃣ Create Widget
# ==========================================

dbutils.widgets.text("CustomerOnboard", "")
CustomerOnboard = dbutils.widgets.get("CustomerOnboard")

dbutils.widgets.text(
    "run_id",
    "472519629468310"
)
run_id = dbutils.widgets.get("run_id")
print(run_id)
print(CustomerOnboard)

In [0]:
# Read Customer Source Code
CustomerSourceCode=""

if CustomerOnboard != "Onboard":
  CustomerSourceCode=CustomerOnboard
else:  

  # Take JSON Code for OnBoarding
  from pyspark.sql.functions import col

  # Read the file by explicitly telling Spark to parse multi-line JSON
  df = spark.read.format('json') \
      .option("multiLine", True) \
      .load('/Volumes/clinicalforge/metadata/onboardingfile/Customer_Onboarding/onboard_customer.json')

  # Flatten nested columns to get 13 individual columns
  df_flattened = df.select(
      "*",
      *[col(f"{nested_col}.*") for nested_col in df.columns if df.schema[nested_col].dataType.typeName() == "struct"]
  ).drop(*[nested_col for nested_col in df.columns if df.schema[nested_col].dataType.typeName() == "struct"])

  # View the flattened data
  #display(df_flattened)
  # Define StoredProcedure Variables
  # 1.Customer Variables
  p_CustomerCode = df_flattened.select('CustomerCode').collect()[0]["CustomerCode"]
  p_CustomerName = df_flattened.select('CustomerName').collect()[0]["CustomerName"]
  p_SubscriptionTier = df_flattened.select('SubscriptionTier').collect()[0]["SubscriptionTier"]

  # 2. ConnectionDetails
  p_HostServer = df_flattened.select('HostServer').collect()[0]["HostServer"]
  p_DatabaseName = df_flattened.select('DatabaseName').collect()[0]["DatabaseName"]
  p_UserName = df_flattened.select('UserName').collect()[0]["UserName"]
  p_Password = df_flattened.select('Password').collect()[0]["Password"]
  p_TargetPlatform=df_flattened.select('TargetPlatform').collect()[0]["TargetPlatform"]

  # 3. EventHub Details
  p_NamespaceName = df_flattened.select('NamespaceName').collect()[0]["NamespaceName"]
  p_TopicName = df_flattened.select('TopicName').collect()[0]["TopicName"]
  p_ConnSecretStringEH = df_flattened.select('ConnSecretStringEH').collect()[0]["ConnSecretStringEH"]

  # 4. Product Details
  p_ProductCode = df_flattened.select('ProductCode').collect()[0]["ProductCode"]
  p_ProductName = df_flattened.select('ProductName').collect()[0]["ProductName"]

  # Execute the procedure and capture any returned output or logs
  df = spark.sql(f"CALL clinicalforge.metadata.sp_ProvisionNewCustomer('{p_CustomerCode}', '{p_CustomerName}', '{p_SubscriptionTier}', '{p_ProductCode}', '{p_ProductName}', '{p_TargetPlatform}', '{p_HostServer}', '{p_DatabaseName}', '{p_UserName}', '{p_Password}', '{p_NamespaceName}', '{p_TopicName}', '{p_ConnSecretStringEH}')")
  df.show()
  # Add CustomerCode to Global Variable
  CustomerSourceCode=p_CustomerCode

  #CustomerOnboard = "Health"

dbutils.jobs.taskValues.set(
    key="CustomerCode_Source",
    value=CustomerSourceCode

)

print(f"CustomerCode Value:{CustomerSourceCode}")
print("Task value 'CustomerCode_Source' has been set.")
#